In [1]:
import pickle
import networkx as nx
import pennylane as qml
from joblib import Parallel, delayed
from pennylane import numpy as np
from itertools import combinations



In [5]:
def make_M_V(P, w):
    M = P @ np.diag(w) @ P.T
    return M, np.diag(P @ w) - M

In [17]:
def make_dicke_state(n, k):
        if k < 0 or k > n:
            raise ValueError("k must satisfy 0 <= k <= n")
    
        state = np.zeros(2**n, dtype=complex)
        weight_k_states = list(combinations(range(n), k))
        
        amplitude = 1 / np.sqrt(len(weight_k_states))

        #convert bit string to integer
        for one_positions in weight_k_states:
            basis_index=0
            for wire in one_positions:
                basis_index+=2**(n-1-wire)
            state[basis_index] = amplitude

        

        d_state = np.array(state, requires_grad=False)
        if (comb(n,k)!=np.count_nonzero(d_state)):
            raise ValueError("amplitude count incorrect")
    
        return d_state

In [21]:
from math import comb
def max_cut_from_Q_hwpm(Q,
                   seed: int,
                   num_layers: int,
                   scaling_constant: float,
                   step_size: float, 
                   max_trials: int,
                   threshold: float,
                  ):
    rng = np.random.default_rng(seed=seed)
    n_wires = Q.shape[0]
    triu_indices = np.triu_indices(Q.shape[0])
    edge_list = []
    for (u, v) in zip(triu_indices[0], triu_indices[1]):
        if u == v:
            continue
        elif Q[u][v] != 0.0:
            edge_list.append((u, v))
    num_edges = len(edge_list)
    print("Number of edges: ", num_edges)
    
    
    beta_params = scaling_constant * rng.random([num_edges, num_layers], requires_grad=True)
    gamma_params = scaling_constant * rng.random([num_edges, num_layers], requires_grad=True)
    
    def U_B(beta):
        edge_idx=0
        for u, v in edge_list:
            theta = beta[edge_idx] 
            qml.S(wires=u) 
            
            qml.SingleExcitationMinus(-2*theta,wires=[u,v])
            qml.adjoint(qml.S)(wires=u) 
            edge_idx+=1
        
        
    def U_C(gamma):
        edge_idx = 0
        for u, v in edge_list: 
            w = Q[u][v]
            qml.CNOT(wires=(u, v))
            qml.RZ(-w * gamma[edge_idx], wires=v)
            qml.CNOT(wires=(u, v))
            edge_idx += 1 
    dev = qml.device('default.qubit', wires=n_wires)

    

    target_weight = n_wires // 2
    initial_state = make_dicke_state(n_wires, target_weight)

    #confirm number of non-zero amps and then confirm they sum to one
    print("Number of nonzero amplitudes:", np.count_nonzero(initial_state))
    print("Predicted number of non-zero amplitudes: ", comb(n_wires, target_weight))
    print("State norm:", np.sum(np.abs(initial_state) ** 2))
        
    @qml.qnode(dev)
    def circuit(gammas, betas):
        qml.StatePrep(initial_state, wires=range(n_wires))
        #for wire in range(n_wires):
        #    qml.Hadamard(wires=wire)
        for k in range(num_layers):
            U_C(gammas[:, k])
            U_B(betas[:, k])
        return [qml.expval(qml.PauliZ(u) @ qml.PauliZ(v)) for u, v in edge_list]
    
    def objective(gammas, betas):
        expectation_values = circuit(gammas, betas)
        expectation_values = np.array(expectation_values)
        values = 0.0
        for u in range(n_wires):
            values += Q[u][u]
        edge_idx = 0
        for u, v in edge_list:
            w  = Q[u][v]
            values += 2 * expectation_values[edge_idx] * w
            edge_idx += 1
        return -values

    opt = qml.AdamOptimizer(stepsize=step_size)
    objective_list = [objective(gamma_params, beta_params)]
    params_list = [(gamma_params, beta_params)]
    for i in range(max_trials):
        (gamma_params, beta_params), current_obj = opt.step_and_cost(objective, gamma_params, beta_params)
        if (i + 1) % 50 == 0:
            print("Current obj: ", current_obj)
            if np.abs(current_obj - objective_list[-1]) <= threshold:
                objective_list.append(current_obj)
                params_list.append((gamma_params, beta_params))
                break
        objective_list.append(current_obj)
        params_list.append((gamma_params, beta_params))
    result = [objective_list, params_list]
    best_params = result[1][np.argmin(result[0])]
    best_qaoa_objective = np.min(result[0])
    
    print("Best param: ", best_params)
    print("Best QAOA obj: ", best_qaoa_objective)
    
    dev_sampling = qml.device('default.qubit', wires=n_wires)
    @qml.set_shots(shots = 10000)
    @qml.qnode(dev_sampling)
    def sampling_circuit(gammas, betas):
        qml.StatePrep(initial_state, wires=range(n_wires))
        #for wire in range(n_wires):
        #    qml.Hadamard(wires=wire)
        for k in range(num_layers):
            U_C(gammas[:, k])
            U_B(betas[:, k])
        return qml.sample()
    best_gammas, best_betas = best_params
    samples = sampling_circuit(best_gammas, best_betas)
    sample_weights = np.sum(samples, axis=1)
    print("Unique sampled Hamming weights:", np.unique(sample_weights))
    print("first 10 samples: ",samples[:10])
    values = []
    formatted_samples = []
    for sample in samples:
        new_x = 2*np.array(sample)-1
        value = new_x@Q@new_x.transpose()
        formatted_samples.append(sample)
        values.append(value)
    print("Best objective:", np.max(values))
    print("Average objective:", np.mean(values))
    print("Best x:", formatted_samples[np.argmax(values)])
    return [np.mean(values), best_qaoa_objective, best_params] #np.max(values), formatted_samples[np.argmax(values)], 





In [15]:

def fair_cut_from_Q_hwpm(Vlist, 
                   seed: int,
                   num_layers: int,
                   scaling_constant: float,
                   step_size: float, 
                   max_trials: int,
                   threshold: float,
                   #non_improving_steps: int
                   ):
    rng = np.random.default_rng(seed=seed)
    hyper_edges_list = []
    for hyper_edge_matrix in Vlist:
        hyper_edge = []
        hyper_edges_list.append(set(np.where(hyper_edge_matrix != 0.0)[1]))
    n_wires = Vlist[0].shape[0]
    edge_list = []
    hyper_edges_clique_enum = []
    hyper_edge_idx = 0
    for hyper_edge in hyper_edges_list:
        hyper_edge_expansion = []
        for edge in combinations(hyper_edge, 2):
            hyper_edge_expansion.append((np.min(edge), np.max(edge), float(Vlist[hyper_edge_idx][np.min(edge)][np.max(edge)])))
        edge_list += hyper_edge_expansion
        hyper_edges_clique_enum.append(hyper_edge_expansion)
        hyper_edge_idx += 1
    print("Len Edge list: ", len(edge_list))
    edge_list = set(edge_list)
    print("Len Set Edge list: ", len(edge_list))
    num_edges = len(edge_list)
    beta_params = scaling_constant * rng.random([num_edges, num_layers], requires_grad=True)
    gamma_params = scaling_constant * rng.random([num_edges, num_layers], requires_grad=True)

    
    def U_B(beta):
        edge_idx=0
        for u, v, _ in edge_list:
            theta = beta[edge_idx] 
            qml.S(wires=u) 
            
            qml.SingleExcitationMinus(-2*theta,wires=[u,v])
            qml.adjoint(qml.S)(wires=u) 
            edge_idx+=1
        # for wire in range(n_wires):
        #     qml.RX(2 * beta[wire], wires=wire)
    def U_C(gamma):
        edge_idx = 0
        for u, v, w in edge_list: 
            qml.CNOT(wires=(u, v))
            qml.RZ(-w * gamma[edge_idx], wires=v)
            qml.CNOT(wires=(u, v))
            edge_idx += 1

    target_weight = n_wires // 2
    initial_state = make_dicke_state(n_wires, target_weight)

    #confirm number of non-zero amps and then confirm they sum to one
    print("Number of nonzero amplitudes:", np.count_nonzero(initial_state))
    print("Predicted number of non-zero amplitudes: ", comb(n_wires, target_weight))
    print("State norm:", np.sum(np.abs(initial_state) ** 2))
    
    dev = qml.device('default.qubit', wires=n_wires)
    @qml.qnode(dev)
    def circuit(gammas, betas):
        qml.StatePrep(initial_state, wires=range(n_wires))
        # for wire in range(n_wires):
        #     qml.Hadamard(wires=wire)
        for k in range(num_layers):
            U_C(gammas[:, k])
            U_B(betas[:, k])
        return [qml.expval(qml.PauliZ(u) @ qml.PauliZ(v)) for u, v, _ in edge_list]
    
    def objective(gammas, betas):
        tau = 0.005
        expectation_values = circuit(gammas, betas)
        exp_values_dict = {}
        edge_idx = 0
        for u, v, _ in edge_list:
            exp_values_dict[f"{u}-{v}"] = expectation_values[edge_idx]
            edge_idx += 1
        all_hyper_edges = []
        for hyper_edge_idx in range(len(hyper_edges_clique_enum)):
            hyper_edge_values = 0.0
            num_vertices = len(hyper_edges_list[hyper_edge_idx])
            for u, v, _ in hyper_edges_clique_enum[hyper_edge_idx]:
                hyper_edge_values += 2 * exp_values_dict[f"{u}-{v}"]
            value = (num_vertices - 1)/num_vertices  - (hyper_edge_values/(num_vertices)**2) #
            #value = np.exp(-value/tau)
            all_hyper_edges.append(value)
        return -np.min(all_hyper_edges) #tau*np.log(np.sum(all_hyper_edges)) #
        #values = np.min(all_hyper_edges)
        #return -values
    opt = qml.AdamOptimizer(stepsize=step_size)
    objective_list = [objective(gamma_params, beta_params)]
    params_list = [(gamma_params, beta_params)]
    for i in range(max_trials):
        (gamma_params, beta_params), current_obj = opt.step_and_cost(objective, gamma_params, beta_params)
        if (i + 1) % 50 == 0:
            print("Current obj: ", current_obj)
            if np.abs(current_obj - objective_list[-1]) <= threshold:
                objective_list.append(current_obj)
                params_list.append((gamma_params, beta_params))
                break
        objective_list.append(current_obj)
        params_list.append((gamma_params, beta_params))
    result = [objective_list, params_list]
    best_params = result[1][np.argmin(result[0])]
    best_qaoa_objective = np.min(result[0])
    
    num_shots = 10000
    dev_sampling = qml.device('default.qubit', wires=n_wires)
    @qml.set_shots(shots = num_shots)
    @qml.qnode(dev_sampling)
    def sampling_circuit(gammas, betas):
        qml.StatePrep(initial_state, wires=range(n_wires))
        # for wire in range(n_wires):
        #     qml.Hadamard(wires=wire)
        for k in range(num_layers):
            U_C(gammas[:, k])
            U_B(betas[:, k])
        return qml.counts()

    best_gammas, best_betas = best_params
    counts = sampling_circuit(best_gammas, best_betas)
    Qlist_values = []
    for Q in Vlist:
        Q_value = 0.0
        for sample in counts.keys():
            new_x = np.array([2*int(sample[i]) - 1 for i in range(len(sample))])
            value = new_x@Q@new_x.transpose()
            Q_value += (counts[sample]/num_shots)*value
        Qlist_values.append(Q_value)
    print("Best fair objective:", np.min(Qlist_values))
    return [np.min(Qlist_values), best_qaoa_objective, best_params]



In [13]:
def run_graph_hwpm(idx, num_layers, run_type, graph_type):
    # if graph_type == "contact-high-school":
    #     if idx < 10: 
    #         hypergraph_name = f"contact-high-school_0{idx}"
    #     else:
    #         hypergraph_name = f"contact-high-school_{idx}"
    # elif graph_type == "email-Enron":
    #     hypergraph_name = f"email-Enron_{idx}"
    # elif graph_type == "congress-bills":
    #     if idx < 10:
    #         hypergraph_name = f"congress-bills_0{idx}"
    #     else:
    #         hypergraph_name = f"congress-bills_{idx}"
    if idx < 10:
        hypergraph_name = f"0{idx}"
    else:
        hypergraph_name = f"{idx}"
    print("Name: ", hypergraph_name)
    P = np.load(f'data/{graph_type}/P/{hypergraph_name}.npy')
    w = np.load(f'data/{graph_type}/w/{hypergraph_name}.npy')
    M, V = make_M_V(P, w)
    Vlist = []
    Q = V
    print(Q)
    #create network X path graph w same number of nodes
    Vlist = [w[i] * (np.diag(P[:, i]) - P[:, i].reshape(P.shape[0], 1) * P[:, i].reshape(1, P.shape[0])) for i in range(P.shape[1])]
    if run_type == "max":
        step_size = 5e-2
    elif run_type == "fair":
        step_size = 5e-2
    threshold = 1e-8
    max_trials = 300
    scaling_constant = 5e-2
    seed = 42
    if run_type == "max":
        results  =  max_cut_from_Q_hwpm(Q=Q,
                                    seed = seed,
                                    num_layers = num_layers,
                                    scaling_constant = scaling_constant,
                                    step_size = step_size, 
                                    max_trials = max_trials,
                                    threshold = threshold)
        print("Best QAOA obj: ", results[1])
        print("Best average max:", results[0])
    elif run_type == "fair":
        results  =  fair_cut_from_Q_hwpm(Vlist = Vlist, #
                                    seed = seed,
                                    num_layers = num_layers,
                                    scaling_constant = scaling_constant,
                                    step_size = step_size, 
                                    max_trials = max_trials,
                                    threshold = threshold)
        print("Best QAOA obj: ", results[1])
        print("Best fair sampling:", results[0])
    return results


In [10]:
graphs=[1,2,3,4,5]

In [11]:
for g in graphs:
    run_graph_hwpm(
    idx=g,
    num_layers=3,
    run_type="max",
    graph_type="poisson_16_12_3"
)

Name:  01
[[ 0.82666667  0.         -0.04        0.          0.         -0.26222222
   0.          0.         -0.33333333  0.         -0.04       -0.15111111]
 [ 0.          0.36111111 -0.02777778  0.         -0.13888889 -0.02777778
  -0.02777778  0.         -0.11111111 -0.02777778  0.          0.        ]
 [-0.04       -0.02777778  1.05611111 -0.125      -0.13888889 -0.06777778
  -0.06777778 -0.165      -0.11111111 -0.13027778 -0.1425     -0.04      ]
 [ 0.          0.         -0.125       0.625       0.         -0.25
   0.         -0.125       0.         -0.0625     -0.0625      0.        ]
 [ 0.         -0.13888889 -0.13888889  0.          1.05555556 -0.13888889
  -0.02777778  0.         -0.33333333 -0.02777778  0.         -0.25      ]
 [-0.26222222 -0.02777778 -0.06777778 -0.25       -0.13888889  1.87527778
  -0.09027778 -0.0625     -0.58333333 -0.13888889 -0.21361111 -0.04      ]
 [ 0.         -0.02777778 -0.06777778  0.         -0.02777778 -0.09027778
   0.48638889 -0.1025      0

In [16]:
for g in graphs:
    run_graph_hwpm(
    idx=g,
    num_layers=3,
    run_type="fair",
    graph_type="poisson_16_12_3"
)

Name:  01
[[ 0.82666667  0.         -0.04        0.          0.         -0.26222222
   0.          0.         -0.33333333  0.         -0.04       -0.15111111]
 [ 0.          0.36111111 -0.02777778  0.         -0.13888889 -0.02777778
  -0.02777778  0.         -0.11111111 -0.02777778  0.          0.        ]
 [-0.04       -0.02777778  1.05611111 -0.125      -0.13888889 -0.06777778
  -0.06777778 -0.165      -0.11111111 -0.13027778 -0.1425     -0.04      ]
 [ 0.          0.         -0.125       0.625       0.         -0.25
   0.         -0.125       0.         -0.0625     -0.0625      0.        ]
 [ 0.         -0.13888889 -0.13888889  0.          1.05555556 -0.13888889
  -0.02777778  0.         -0.33333333 -0.02777778  0.         -0.25      ]
 [-0.26222222 -0.02777778 -0.06777778 -0.25       -0.13888889  1.87527778
  -0.09027778 -0.0625     -0.58333333 -0.13888889 -0.21361111 -0.04      ]
 [ 0.         -0.02777778 -0.06777778  0.         -0.02777778 -0.09027778
   0.48638889 -0.1025      0

In [22]:
run_graph_hwpm(
    idx=1,
    num_layers=1,
    run_type="max",
    graph_type="poisson_16_12_3"
)

Name:  01
[[ 0.82666667  0.         -0.04        0.          0.         -0.26222222
   0.          0.         -0.33333333  0.         -0.04       -0.15111111]
 [ 0.          0.36111111 -0.02777778  0.         -0.13888889 -0.02777778
  -0.02777778  0.         -0.11111111 -0.02777778  0.          0.        ]
 [-0.04       -0.02777778  1.05611111 -0.125      -0.13888889 -0.06777778
  -0.06777778 -0.165      -0.11111111 -0.13027778 -0.1425     -0.04      ]
 [ 0.          0.         -0.125       0.625       0.         -0.25
   0.         -0.125       0.         -0.0625     -0.0625      0.        ]
 [ 0.         -0.13888889 -0.13888889  0.          1.05555556 -0.13888889
  -0.02777778  0.         -0.33333333 -0.02777778  0.         -0.25      ]
 [-0.26222222 -0.02777778 -0.06777778 -0.25       -0.13888889  1.87527778
  -0.09027778 -0.0625     -0.58333333 -0.13888889 -0.21361111 -0.04      ]
 [ 0.         -0.02777778 -0.06777778  0.         -0.02777778 -0.09027778
   0.48638889 -0.1025      0

[np.float64(14.578756666666665),
 np.float64(-14.568946201925913),
 (tensor([[ 10.9006661 ],
          [  0.2625947 ],
          [  3.67308406],
          [ -7.53980451],
          [  0.9447752 ],
          [ -4.1685188 ],
          [ -0.89522495],
          [ -4.6847404 ],
          [ -2.46980666],
          [  8.5379472 ],
          [ -6.70912042],
          [  1.08586592],
          [ -0.04359005],
          [ -3.80267461],
          [ -2.26776882],
          [  0.39275986],
          [ -9.48880895],
          [  0.87989566],
          [  0.4452754 ],
          [  1.64747313],
          [  6.76225966],
          [  2.28083563],
          [  0.53694501],
          [ -0.76696581],
          [ -0.49823622],
          [  3.90774678],
          [  6.08468396],
          [ -5.28774283],
          [  0.65765555],
          [ -2.8752441 ],
          [ -5.56685203],
          [  2.17476578],
          [ -0.77298914],
          [ -0.86219297],
          [ -2.97968532],
          [ -0.33648295